In [ ]:
import pandas as pd
from transformers import AutoTokenizer, AutoModelForSeq2SeqLM
from functools import partial
import nltk
from nltk.tokenize import sent_tokenize

import torch

DEVICE = 'cuda' if torch.cuda.is_available() else 'cpu'

# Some constants
dataset_path = "../resources/dataset"
dataset = 'youtoxic_english_1000.csv'
augmented_dataset = f"augmented_{dataset}"
original_dataset = f"{dataset_path}/{dataset}"

# Array of columns to use for augmenting dataset
targets = [
    'IsToxic', 
    'IsAbusive', 
    'IsProvocative', 
    'IsObscene', 
    'IsHatespeech', 
    'IsRacist',
    'IsThreat',
    'IsReligiousHate',
    'IsNationalist'
]

# Load original dataset
df = pd.read_csv(original_dataset)
df_work = df.drop_duplicates(subset=['Text'])

# Download necessary NLTK data for sentence tokenization
nltk.download('punkt')

# --- Hugging Face Model Loading (Keep as is) ---
MODEL_NAME_EN_MUL = 'Helsinki-NLP/opus-mt-en-mul'
tokenizer_en_mul = AutoTokenizer.from_pretrained(MODEL_NAME_EN_MUL)
model_en_mul = AutoModelForSeq2SeqLM.from_pretrained(MODEL_NAME_EN_MUL)

MODEL_NAME_MUL_EN = "Helsinki-NLP/opus-mt-mul-en"
tokenizer_mul_en = AutoTokenizer.from_pretrained(MODEL_NAME_MUL_EN)
model_mul_en = AutoModelForSeq2SeqLM.from_pretrained(MODEL_NAME_MUL_EN)

model_en_mul.to(DEVICE)
model_mul_en.to(DEVICE)

print("MarianMT Models (EN->MUL) loaded.")
# -----------------------------------------------

# The core translation function now handles only a single sentence.
def back_translate_single_sentence(sentence, lan):
    """
    Performs back-translation (English -> Foreign -> English) for a single sentence.
    """
    if not sentence or not isinstance(sentence, str) or not sentence.strip():
        return sentence

    try:
        SAMPLING_KWARGS = {
            'do_sample': True,
            'top_k': 50,
            'temperature': 0.9,
            'num_return_sequences': 1 # We still only want one output sequence
        }
        
        # 1. Translate EN -> Foreign
        text_with_tag = f">>{lan}<< " + sentence
        
        inputs_en_mul = tokenizer_en_mul(text_with_tag, return_tensors="pt").to(DEVICE)
        outputs_en_mul = model_en_mul.generate(**inputs_en_mul, **SAMPLING_KWARGS)
        text_mul = tokenizer_en_mul.decode(outputs_en_mul[0], skip_special_tokens=True)
        
        # 2. Translate Foreign -> EN (Back-translation)
        inputs_mul_en = tokenizer_mul_en(text_mul, return_tensors="pt").to(DEVICE)
        outputs_mul_en = model_mul_en.generate(**inputs_mul_en, **SAMPLING_KWARGS)
        text_en_back = tokenizer_mul_en.decode(outputs_mul_en[0], skip_special_tokens=True)
        
        # We don't print here to avoid excessive logging inside the loop
        return text_en_back
    
    except Exception as e:
        # If translation fails, return the original sentence to prevent data loss
        print(f"⚠️ Error when local translating sentence: '{sentence[:30]}...': {e}")
        return sentence

# The new function that orchestrates the sentence-level process.
def augment_by_sentence(full_text, lan):
    """
    Splits the full text into sentences, back-translates each one, 
    and rejoins them into a single augmented string.
    """
    if not full_text or not isinstance(full_text, str):
        return full_text
        
    # Split the full text into individual sentences
    sentences = sent_tokenize(full_text)
    augmented_sentences = []
    
    for sentence in sentences:
        # Use the single-sentence translator
        back_translated_sentence = back_translate_single_sentence(sentence, lan)
        augmented_sentences.append(back_translated_sentence)
        
    # Recombine the augmented sentences, preserving spacing structure
    return " ".join(augmented_sentences)


# --- DataFrame Processing ---

LANGS_TO_USE = [
    'es', 
    # 'de', 
    # 'fr', 
    # 'ru', 
    # 'ar'
]

# List to keep original dataframe and the augmented
augmented_dfs = []

original_df = df_work.copy()
original_df['Augmentation_Type'] = 'Original'
augmented_dfs.append(original_df)

print(f"Original dataset size: {len(df)}")

combined_mask = pd.Series(False, index=df.index)
# Iterate through the targets and use the OR operator (|) to combine them
for col in targets:
    # The '|' operator combines the current mask with the next column's boolean Series
    combined_mask = combined_mask | df_work[col]
    
# Filter the rows to be augmented
tagged_comments = df_work[combined_mask].copy()

# The augmentation loop 
for lan in LANGS_TO_USE:
    print(f"\n🔄 Augmenting data with: {lan}...")
    
    temp_df = tagged_comments.copy()
    
    # Use partial to pre-fill the language for the augmentation function
    sentence_back_translator = partial(augment_by_sentence, lan=lan)
    
    # Apply the new sentence-level augmentation function to the 'Text' column
    # This will take longer because of the extra processing steps.
    temp_df['Text'] = temp_df['Text'].apply(sentence_back_translator)
    
    temp_df['Augmentation_Type'] = f'BT_{lan}'
    
    augmented_dfs.append(temp_df)
    
    print(f"    - Added Rows: {len(temp_df)}")
    
    
final_augmented_df = pd.concat(augmented_dfs, ignore_index=True)

print("Done!.")

# Save new dataset
final_augmented_df.to_csv(f"{dataset_path}/{augmented_dataset}")

cpu


[nltk_data] Downloading package punkt to /home/vscode/nltk_data...
[nltk_data]   Package punkt is already up-to-date!


MarianMT Models (EN->MUL) loaded.
Original dataset size: 1000

🔄 Augmenting data with: es...


/tmp/ipykernel_53589/605196264.py:137: UserWarning: Boolean Series key will be reindexed to match DataFrame index.
  tagged_comments = df_work[combined_mask].copy()


KeyboardInterrupt: 